In [ ]:
!pip uninstall opencv-pytho
!pip uninstall opencv-python opencv-contrib-python -y
!pip install opencv-python==4.10.0.84
!pip install opencv-contrib-python==4.10.0.84

In [ ]:
import torch
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms as T
from PIL import Image
import cv2
cap = cv2.VideoCapture(0)  # Test with the webcam
if cap.isOpened():
    print("VideoCapture works!")
else:
    print("Error: Unable to open video.")
cap.release()
import numpy as np
import matplotlib.pyplot as plt

def load_model(model_path):
    num_classes = 3  # Adjust according to your dataset
    # Create a ResNet-101 backbone with FPN
    backbone = resnet_fpn_backbone('resnet101', pretrained=True)
    model = FasterRCNN(backbone, num_classes=num_classes)
    
    # Load model weights
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model

def predict_frame(model, frame, confidence_threshold=0.1):
    # Convert OpenCV frame (BGR) to PIL Image (RGB)
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
    # Transform the image
    transform = T.Compose([T.ToTensor()])
    img_tensor = transform(image)
    
    # Get prediction
    with torch.no_grad():
        prediction = model([img_tensor])
    
    boxes = prediction[0]['boxes'].cpu().numpy()
    scores = prediction[0]['scores'].cpu().numpy()
    labels = prediction[0]['labels'].cpu().numpy()
    
    # Filter based on confidence threshold
    keep = scores > confidence_threshold
    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]
    
    return boxes, scores, labels

def visualize_predictions_on_frame(frame, boxes, scores, labels, class_names=None):
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box.astype(int)
        
        label_text = f"Class: {labels[i]}"
        if class_names and 0 <= labels[i] < len(class_names):
            label_text = f"{class_names[labels[i]]}"
        
        score_text = f"{scores[i]:.2f}"
        
        # Draw rectangle
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red box
        
        # Draw label and score
        cv2.putText(frame, f"{label_text}: {score_text}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
    
    return frame

def process_video(video_path, model, output_path, class_names=None, confidence_threshold=0.5):
    # Open video file
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Set up Matplotlib
    plt.ion()  # Turn on interactive mode
    fig, ax = plt.subplots(figsize=(10, 6))
    img_display = ax.imshow(np.zeros((height, width, 3), dtype=np.uint8))  # Placeholder image
    ax.axis("off")
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Run predictions
        boxes, scores, labels = predict_frame(model, frame, confidence_threshold)
        
        # Visualize predictions
        frame = visualize_predictions_on_frame(frame, boxes, scores, labels, class_names)
        
        # Write the frame to the output video
        out.write(frame)
        
        # Display the frame using Matplotlib
        img_display.set_data(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        fig.canvas.draw()
        fig.canvas.flush_events()
    
    cap.release()
    out.release()
    plt.ioff()  # Turn off interactive mode
    plt.close(fig)
    print("Video processing complete. Output saved to:", output_path)

def process_video_live(video_path, model, class_names=None, confidence_threshold=0.5):
    # Open video file
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print("Error: Unable to open video file.")
        return

    target_width, target_height = 640, 480  

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # Resize the frame to Full HD
        frame = cv2.resize(frame, (target_width, target_height))
        
        # Run predictions
        boxes, scores, labels = predict_frame(model, frame, confidence_threshold)
        
        # Visualize predictions
        frame = visualize_predictions_on_frame(frame, boxes, scores, labels, class_names)
        
        # Display the frame in real-time
        cv2.imshow("Scorpion Detection", frame)
        
        # Exit the display when the user presses the 'q' key
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    print("Video processing complete.")

if __name__ == "__main__":
    model_path = "fasterrcnn_resnet50_epoch_20_final.pth"
    video_path = "test-video/video3.mp4"  # Replace with your video path
    output_path = "Result_scorpion_detection.mp4"
    
    # Replace with your class names
    class_names = [
        "background", "scorpion",
    ]
    
    # Load the entire model
    model = torch.load(model_path, weights_only=False)
    model.eval()
    
    # Process the video
    #process_video(video_path, model, output_path, class_names, confidence_threshold=0.8)
    process_video_live(video_path, model, class_names, confidence_threshold=0.8)
    


VideoCapture works!
Video processing complete.
